In [ ]:
import json
import os
import time
import warnings
from dotenv import load_dotenv
from google import genai
from google.genai import types
import pandas as pd
from pydantic import BaseModel
from tqdm import tqdm
from typing_extensions import List

warnings.filterwarnings("ignore")
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Load CSV
input_file = (
    "../content_libraries/netflix_samples/netflix_reviews_2022_sample.csv"
)
output_file = "netflix_reviews_2022_sample_completed.csv"
df = pd.read_csv(input_file)

if "annotation" not in df.columns:
  df["annotation"] = None

# Extract manual examples
labeled_df = df[
    df["annotation"].notna() & (df["annotation"].astype(str).str.strip() != "")
]
# Use a smaller, diverse sample for examples to keep prompt light
sample_size = min(20, len(labeled_df))
examples_sample = labeled_df.sample(n=sample_size, random_state=42)

examples_text = "\n".join([
    f"Review: \"{row['content_clean'] if 'content_clean' in row and pd.notna(row['content_clean']) else row['content']}\" | Rating: {row['score']} -> Annotation:"
    f" {row['annotation']}"
    for _, row in examples_sample.iterrows()
])


# Simplified Schema using an integer index (1 to N)
class ItemResult(BaseModel):
  item_num: int
  annotation: str


class BatchResponse(BaseModel):
  results: List[ItemResult]

# System Instruction
SYSTEM_INSTRUCTION = f"""
You are an expert annotating customer reviews for Netflix and Amazon Prime Video.
Your job is to annotate multiple reviews at once using shorthand codes.

TOPIC CODES:
- c: Catalog (content selection, variety, availability)
- p: Pricing (subscription cost, ads, value for money)
- a: App functionality (bugs, crashes, streaming issues, UI)
- m: Review meant for specific movie/show
- u: Unspecified (general praise/complaint, e.g. "Great app", "I love it")
- o: Other (account issues, customer service, etc.)

SENTIMENT CODES:
- + : Positive
- 0 : Neutral (or genuine balance)
- - : Negative

RULES:
1. Match review_id exactly as provided.
2. For annotation, output ONLY shorthand codes separated by spaces (e.g. "c+", "u+ p-", "a-").
3. Follow the logic shown in these human-annotated examples:
{examples_text}
"""


# Batch Annotation Function
def annotate_batch(batch_df, max_retries=5):
  items = []
  for idx, (_, r) in enumerate(batch_df.iterrows(), start=1):
    text = (
        r["content_clean"]
        if "content_clean" in r and pd.notna(r["content_clean"])
        else r["content"]
    )
    # Truncate long reviews to avoid hitting token limits in combined prompt
    items.append(
        {"item_num": idx, "score": int(r["score"]), "text": str(text)[:500]}
    ) 

  user_prompt = f"Annotate these {len(items)} reviews:\n" + json.dumps(items)

  for attempt in range(max_retries):
    try:
      response = client.models.generate_content(
          model="gemini-3.1-flash-lite",
          contents=user_prompt,
          config=types.GenerateContentConfig(
              system_instruction=SYSTEM_INSTRUCTION,
              temperature=0.0,
              response_mime_type="application/json",
              response_schema=BatchResponse,
          ),
      )
      data = json.loads(response.text)
      return {item["item_num"]: item["annotation"] for item in data["results"]}
    except Exception as e:
      wait_time = (attempt + 1) * 7  # Waits 7s, 14s, 21s, 28s, 35s
      print(f"\n[Server Hiccup] Retrying in {wait_time}s... (Error: {e})")
      time.sleep(wait_time)

  print(f"\n[Batch Failed] Failed to process a batch after {max_retries} attempts.")
  return {} # Return empty if all retries fail


# === Main Processing Loop ===
MAX_PASSES = 3  # Maximum number of times to try annotating remaining reviews
BATCH_SIZE = 30 # Batch size to keep requests low

for current_pass in range(MAX_PASSES):
    # Find all currently unannotated reviews
    unlabeled_mask = df["annotation"].isna() | (
        df["annotation"].astype(str).str.strip() == ""
    )
    unlabeled_indices = df[unlabeled_mask].index.tolist()

    if not unlabeled_indices:
        print(f"\nPass {current_pass + 1}: All reviews are now annotated!")
        break # Exit if nothing left to do

    total_batches = (len(unlabeled_indices) + BATCH_SIZE - 1) // BATCH_SIZE

    print(
        f"\n--- Pass {current_pass + 1}/{MAX_PASSES}: Annotating {len(unlabeled_indices)} reviews in"
        f" {total_batches} batches ---"
    )

    for i in tqdm(range(0, len(unlabeled_indices), BATCH_SIZE), total=total_batches):
        batch_indices = unlabeled_indices[i : i + BATCH_SIZE]
        batch_df = df.loc[batch_indices]

        results = annotate_batch(batch_df)

        # Map 1..N back to exact dataframe row positions
        for item_num, annotation in results.items():
            if 1 <= item_num <= len(batch_indices):
                target_idx = batch_indices[item_num - 1]
                df.at[target_idx, "annotation"] = annotation

        # Save progress after every batch to disk
        df.to_csv(output_file, index=False)
        time.sleep(1) # Small pause to be gentle on API

    # Check how many are still unannotated after this pass
    remaining_unlabeled_after_pass = df[df["annotation"].isna() | (df["annotation"].astype(str).str.strip() == "")].shape[0]
    if remaining_unlabeled_after_pass == 0:
        print(f"\nPass {current_pass + 1}: All reviews annotated successfully!")
        break
    else:
        print(f"\nPass {current_pass + 1}: {remaining_unlabeled_after_pass} reviews still unannotated. Retrying in next pass.")
        if current_pass + 1 < MAX_PASSES:
            time.sleep(10) # Longer pause before starting a new pass for server cooldown

if remaining_unlabeled_after_pass > 0:
    print(f"\nWarning: Could not annotate {remaining_unlabeled_after_pass} reviews after {MAX_PASSES} passes. Check your data and API status.")
else:
    print(f"\nFinished! All reviews successfully annotated and saved to {output_file}")


--- Pass 1/3: Annotating 979 reviews in 33 batches ---


  0%|          | 0/33 [00:00<?, ?it/s]


[Server Hiccup] Retrying in 7s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})


 21%|██        | 7/33 [02:35<05:57, 13.75s/it]


[Server Hiccup] Retrying in 7s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})


...


[Server Hiccup] Retrying in 7s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})

[Server Hiccup] Retrying in 14s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})


 94%|█████████▍| 31/33 [15:07<01:10, 35.14s/it]


[Server Hiccup] Retrying in 7s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})

[Server Hiccup] Retrying in 14s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})

[Server Hiccup] Retrying in 21s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})

[Server Hiccup] Retrying in 28s... (Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})


100%|██████████| 33/33 [16:49<00:00, 30.59s/it]


Pass 1: All reviews annotated successfully!

Finished! All reviews successfully annotated and saved to netflix_reviews_2022_sample_completed.csv
